#D'abord un petit voyage dans le temps
Executez le code ci-dessous. Les traductions sont-elles correctes ?

Ajoutez le mot « black » (noir) au dictionnaire et à la liste des adjectifs. Testez :
The black cat sees a small bird.
Le système gère-t-il l'accord en genre ?

Modifiez le code pour que « le » devienne « la » devant un nom féminin.
(Indice : ajoutez un dictionnaire genres = {"chat": "m", "maison": "f", ...} et une liste de noms féminins.)

Testez cette phrase : The cat eats.
Que se passe-t-il?

In [ ]:
# --- Dictionnaire EN → FR ---
dictionnaire = {
    "the": "le",
    "a": "un",
    "cat": "chat",
    "dog": "chien",
    "fish": "poisson",
    "bird": "oiseau",
    "eats": "mange",
    "sees": "voit",
    "big": "gros",
    "small": "petit",
    "black": "noir"
}

# --- Adjectifs (pour la règle d'inversion) ---
adjectifs = {"big", "small"}

def traduire_rbmt(phrase):
    """
    Traduction à base de règles :
    1. Segmentation
    2. Traduction mot à mot
    3. Règle d'inversion Adjectif + Nom
    """
    mots = phrase.lower().replace(".", "").split()
    traduction = [dictionnaire.get(m, m) for m in mots]

    # Règle : inverser Adjectif + Nom
    resultat = []
    i = 0
    while i < len(traduction):
        # Si on a un adjectif suivi d'un nom
        if (mots[i] in adjectifs and i + 1 < len(mots)
            and mots[i+1] not in adjectifs):
            resultat.append(traduction[i+1])  # nom
            resultat.append(traduction[i])    # adjectif
            i += 2
        else:
            resultat.append(traduction[i])
            i += 1

    return " ".join(resultat)

# --- Test ---
phrases = [
    "The big cat eats a small fish",
    "The dog sees a bird",
]

for p in phrases:
    print(f"EN : {p}")
    print(f"FR : {traduire_rbmt(p)}")
    print()

EN : The big cat eats a small fish
FR : le chat gros mange un poisson petit

EN : The dog sees a bird
FR : le chien voit un oiseau



# Alignement d'un corpus parallèle littéraire français–yiddish–russe

**Objectifs:**
- tester plusieurs librairies d'alignement de mots et de phrases sur un corpus trilingue réel ;
- traiter un cas concret d'écriture de droite à gauche (le yiddish, alphabet hébraïque) ;
- réfléchir aux problèmes de **segmentation en phrases** propres à un texte littéraire, et à des façons de les surmonter.

Ce notebook fournit un **exemple-jouet** (5 phrases inventées, non tirées d'une œuvre existante) pour que le code tourne tout de suite. Remplacez-le par vos propres fichiers dès que possible (section 0).


## 0. Charger votre corpus

Déposez trois fichiers `.txt` (un par langue, une unité de texte par ligne — phrase, réplique ou paragraphe selon votre découpage d'origine) via le panneau **Fichiers** de Colab, ou montez votre Google Drive. Le code ci-dessous cherche `fr.txt` / `yi.txt` / `ru.txt` ; s'il ne les trouve pas, il utilise l'exemple-jouet.

In [ ]:
import os

def load_or_toy(path, toy):
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            return f.read()
    print(f"[info] {path} introuvable — utilisation de l'exemple-jouet.")
    return toy

# --- Exemple-jouet : 5 phrases inventées, PAS tirées d'une œuvre existante ---
# Le yiddish ci-dessous est simplifié pour la démonstration : remplacez-le par
# votre vrai corpus dès que possible, ne vous fiez pas à sa qualité littéraire.

toy_fr = """— Tu viens ? demanda-t-elle. Il ne répondit pas.
Le vieux magasin sentait la poussière et le savon noir ; personne n'y entrait plus depuis la guerre.
« Attends, dit-il, je reviens tout de suite. »
Elle hocha la tête, puis sourit — un sourire triste, presque effrayé.
Dehors, la neige tombait sans bruit sur la ville endormie."""

toy_yi = """— קומסטו? האָט זי געפֿרעגט. ער האָט נישט געענטפֿערט.
דער אַלטער קראָם האָט גערוכן מיט שטויב און שוואַרצער זייף; קיינער איז דאָרטן נישט אַרײַנגעגאַנגען זינט דער מלחמה.
„וואַרט, — האָט ער געזאָגט, — איך קום באַלד צוריק.”
זי האָט געטוסט מיטן קאָפּ, דערנאָך געשמייכלט — אַ טרויעריקער שמייכל, כּמעט דערשראָקן.
דרויסן איז שטיל געפֿאַלן שניי אויף דער פֿאַרשלאָפֿענער שטאָט."""

toy_ru = """— Ты идёшь? — спросила она. Он не ответил.
Старая лавка пахла пылью и чёрным мылом; туда никто не заходил со времён войны.
«Подожди, — сказал он, — я сейчас вернусь».
Она кивнула, потом улыбнулась — грустной, почти испуганной улыбкой.
На улице беззвучно падал снег на уснувший город."""

text_fr = load_or_toy("fr.txt", toy_fr)
text_yi = load_or_toy("yi.txt", toy_yi)
text_ru = load_or_toy("ru.txt", toy_ru)

print(text_fr[:100])

[info] fr.txt introuvable — utilisation de l'exemple-jouet.
[info] yi.txt introuvable — utilisation de l'exemple-jouet.
[info] ru.txt introuvable — utilisation de l'exemple-jouet.
— Tu viens ? demanda-t-elle. Il ne répondit pas.
Le vieux magasin sentait la poussière et le savon n


## 1. Le problème de la segmentation en phrases

**Avant de continuer, répondez à ces deux questions :**
1. Dans vos trois langues, quelles conventions typographiques changent pour marquer le dialogue ou la citation ?
2. Si une règle de segmentation "trop simple" donne 5 phrases en français mais 6 en russe pour le même passage, qu'est-ce que cela signifie concrètement, et pourquoi est-ce un problème pour l'alignement qui suit ?

In [ ]:
#@title Voir la réponse { display-mode: "form" }
"""Un texte littéraire pose des difficultés qu'un corpus administratif ou technique n'a pas :
- **dialogue** introduit par un tiret cadratin (`—`) plutôt qu'une majuscule + point ;
- **guillemets** différents selon la langue (`« »` en français, `„ ”` en yiddish, `« »` ou parfois `„ “` en russe) ;
- **incises** (`dit-il`, `сказала она`) qui coupent une réplique en plusieurs morceaux logiques ;
- **abréviations et points de suspension** confondus avec des fins de phrase ;
- un découpage d'origine (par le traducteur, l'éditeur) qui ne correspond pas forcément d'une langue à l'autre — une phrase française peut devenir deux phrases en russe, ou l'inverse.
"""

In [ ]:
import re

def naive_split(text):
    # découpe sur . ! ? — ignore les abréviations, les guillemets, les tirets de dialogue
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text.replace("\n", " ")) if s.strip()]

for name, text in [("FR", text_fr), ("YI", text_yi), ("RU", text_ru)]:
    sents = naive_split(text)
    print(f"{name}: {len(sents)} « phrases » trouvées")
    for s in sents:
        print("   ", s)
    print()

FR: 7 « phrases » trouvées
    — Tu viens ?
    demanda-t-elle.
    Il ne répondit pas.
    Le vieux magasin sentait la poussière et le savon noir ; personne n'y entrait plus depuis la guerre.
    « Attends, dit-il, je reviens tout de suite.
    » Elle hocha la tête, puis sourit — un sourire triste, presque effrayé.
    Dehors, la neige tombait sans bruit sur la ville endormie.

YI: 6 « phrases » trouvées
    — קומסטו?
    האָט זי געפֿרעגט.
    ער האָט נישט געענטפֿערט.
    דער אַלטער קראָם האָט גערוכן מיט שטויב און שוואַרצער זייף; קיינער איז דאָרטן נישט אַרײַנגעגאַנגען זינט דער מלחמה.
    „וואַרט, — האָט ער געזאָגט, — איך קום באַלד צוריק.” זי האָט געטוסט מיטן קאָפּ, דערנאָך געשמייכלט — אַ טרויעריקער שמייכל, כּמעט דערשראָקן.
    דרויסן איז שטיל געפֿאַלן שניי אויף דער פֿאַרשלאָפֿענער שטאָט.

RU: 7 « phrases » trouvées
    — Ты идёшь?
    — спросила она.
    Он не ответил.
    Старая лавка пахла пылью и чёрным мылом; туда никто не заходил со времён войны.
    «Подожди, — сказал он, — я се

Vous devriez observer un **nombre de "phrases" différent selon la langue**, alors que le texte contient les mêmes 5 unités de sens. C'est exactement le problème annoncé : la ponctuation seule ne suffit pas.

## 2. Une segmentation plus robuste (et ses limites)

`nltk` sait segmenter le français et le russe (modèles Punkt entraînés pour ces langues), mais **pas le yiddish** : il n'existe pas de modèle Punkt pour cette langue. C'est un exemple très concret du manque d'outillage pour les langues peu dotées, même pour une tâche aussi "basique" que couper un texte en phrases.

In [ ]:
!pip -q install nltk
import nltk
nltk.download("punkt", quiet=True)

True

In [ ]:
!pip -q install nltk
import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

from nltk.tokenize import sent_tokenize

print("FR (Punkt):", sent_tokenize(text_fr, language="french"))
print()
print("RU (Punkt):", sent_tokenize(text_ru, language="russian"))
print()
try:
    print("YI (Punkt):", sent_tokenize(text_yi, language="yiddish"))
except Exception as e:
    print("YI (Punkt) — échec attendu :", e)

FR (Punkt): ['— Tu viens ?', 'demanda-t-elle.', 'Il ne répondit pas.', "Le vieux magasin sentait la poussière et le savon noir ; personne n'y entrait plus depuis la guerre.", '« Attends, dit-il, je reviens tout de suite.', '»\nElle hocha la tête, puis sourit — un sourire triste, presque effrayé.', 'Dehors, la neige tombait sans bruit sur la ville endormie.']

RU (Punkt): ['— Ты идёшь?', '— спросила она.', 'Он не ответил.', 'Старая лавка пахла пылью и чёрным мылом; туда никто не заходил со времён войны.', '«Подожди, — сказал он, — я сейчас вернусь».', 'Она кивнула, потом улыбнулась — грустной, почти испуганной улыбкой.', 'На улице беззвучно падал снег на уснувший город.']

YI (Punkt) — échec attendu : 
**********************************************************************
  Resource punkt_tab not found.
  Please use the NLTK Downloader to obtain the resource:

  >>> import nltk
  >>> nltk.download('punkt_tab')
  
  For more information see: https://www.nltk.org/data.html

  Attempted to

### Une alternative indépendante de la langue : `wtpsplit`

[`wtpsplit`](https://github.com/segment-any-text/wtpsplit) segmente en phrases avec un modèle neuronal multilingue, **sans avoir besoin d'un modèle par langue** — donc utilisable tel quel sur le yiddish.

In [ ]:
!pip -q install wtpsplit
from wtpsplit import SaT

sat = SaT("sat-3l-sm")  # petit modèle, suffisant pour ce TP

for name, text in [("FR", text_fr), ("YI", text_yi), ("RU", text_ru)]:
    sents = sat.split(text.replace("\n", " "))
    print(f"{name}: {len(sents)} phrases")
    for s in sents:
        print("   ", s.strip())
    print()

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.4/155.4 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.7/148.7 kB 6.5 MB/s eta 0:00:00


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/965 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  855MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

FR: 6 phrases
    — Tu viens ? demanda-t-elle.
    Il ne répondit pas.
    Le vieux magasin sentait la poussière et le savon noir ; personne n'y entrait plus depuis la guerre.
    « Attends, dit-il, je reviens tout de suite. »
    Elle hocha la tête, puis sourit — un sourire triste, presque effrayé.
    Dehors, la neige tombait sans bruit sur la ville endormie.

YI: 6 phrases
    — קומסטו? האָט זי געפֿרעגט.
    ער האָט נישט געענטפֿערט.
    דער אַלטער קראָם האָט גערוכן מיט שטויב און שוואַרצער זייף; קיינער איז דאָרטן נישט אַרײַנגעגאַנגען זינט דער מלחמה.
    „וואַרט, — האָט ער געזאָגט, — איך קום באַלד צוריק.”
    זי האָט געטוסט מיטן קאָפּ, דערנאָך געשמייכלט — אַ טרויעריקער שמייכל, כּמעט דערשראָקן.
    דרויסן איז שטיל געפֿאַלן שניי אויף דער פֿאַרשלאָפֿענער שטאָט.

RU: 6 phrases
    — Ты идёшь? — спросила она.
    Он не ответил.
    Старая лавка пахла пылью и чёрным мылом; туда никто не заходил со времён войны.
    «Подожди, — сказал он, — я сейчас вернусь».
    Она кивнула, потом улыбнулас

**Question :** `wtpsplit` retrouve-t-il les mêmes 5 unités dans les trois langues ? Sur votre vrai corpus (plus long et plus varié), pensez-vous que ce sera toujours le cas ? Que feriez-vous des cas où deux langues ne s'alignent toujours pas 1 pour 1 après cette étape ? (Gardez cette question en tête pour la section 4.)

## 3. Le yiddish et l'écriture de droite à gauche (RTL)

Le yiddish s'écrit dans l'alphabet hébreu, de droite à gauche. Un notebook, un terminal ou une bibliothèque de visualisation qui ne gère pas le bidirectionnel (bidi) peut afficher le texte dans le mauvais ordre visuel, surtout dès qu'il y a de la ponctuation latine, des chiffres, ou un mot français/russe cité au milieu.

In [ ]:
!pip -q install python-bidi
from bidi import get_display
from IPython.display import display, HTML

exemple = toy_yi.split("\n")[0]
print("Chaîne brute (ordre logique, tel que stocké) :")
print(exemple)
print()
print("Après get_display() (ordre visuel, pour un afficheur qui ne gère pas le bidi) :")
print(get_display(exemple))
print()
print("Avec un vrai rendu HTML dir='rtl' (la bonne solution dans un notebook) :")
display(HTML(f"<p style='direction:rtl; text-align:right; font-size:1.2em'>{exemple}</p>"))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 7.9 MB/s eta 0:00:00
Chaîne brute (ordre logique, tel que stocké) :
— קומסטו? האָט זי געפֿרעגט. ער האָט נישט געענטפֿערט.

Après get_display() (ordre visuel, pour un afficheur qui ne gère pas le bidi) :
.טרעֿפטנעעג טשינ טָאה רע .טגערֿפעג יז טָאה ?וטסמוק —

Avec un vrai rendu HTML dir='rtl' (la bonne solution dans un notebook) :


**Questions :**
1. Quelle différence voyez-vous entre les trois affichages ci-dessus ?
2. Un outil d'alignement affiche en général les paires (mot_source, mot_cible) par leur **indice** dans la phrase, pas par leur position visuelle à l'écran. Pourquoi le bidi complique-t-il quand même la lecture et la vérification manuelle d'un alignement, même si les indices restent corrects ?
3. Si vous deviez construire une visualisation "deux rangées de mots reliées par des traits" (source/cible) pour une paire français–yiddish, que faudrait-il faire pour qu'elle soit lisible ?

## 4. Alignement de mots : tester plusieurs librairies

On travaille maintenant phrase par phrase (utilisez la segmentation obtenue en section 2). Chaque phrase est tokenisée en mots séparés par des espaces — c'est suffisant pour le français, le russe et le yiddish, qui séparent les mots par des espaces (contrairement au chinois, par exemple).

In [ ]:
fr_sents = naive_split(text_fr)
yi_sents = naive_split(text_yi)
ru_sents = naive_split(text_ru)

fr_tok = [s.split() for s in fr_sents]
yi_tok = [s.split() for s in yi_sents]
ru_tok = [s.split() for s in ru_sents]

print(fr_tok[1])
print(ru_tok[1])

['demanda-t-elle.']
['—', 'спросила', 'она.']


### 4a. SimAlign — alignement par embeddings contextuels (mBERT), sans entraînement

In [ ]:
!pip -q install simalign
from simalign import SentenceAligner

aligner = SentenceAligner(model="bert", token_type="bpe", matching_methods="mai")

i = 1  # la phrase "le vieux magasin..."
result = aligner.get_word_aligns(fr_tok[i], ru_tok[i])
print("FR-RU (mwmf) :", result["mwmf"])

result_yi = aligner.get_word_aligns(fr_tok[i], yi_tok[i])
print("FR-YI (mwmf) :", result_yi["mwmf"])

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  714MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

2026-09-29 13:49:29,775 - simalign.simalign - INFO - Initialized the EmbeddingLoader with model: bert-base-multilingual-cased
INFO:simalign.simalign:Initialized the EmbeddingLoader with model: bert-base-multilingual-cased


NameError: name 'fr_tok' is not defined

In [ ]:
# Petite visualisation texte, avec bidi correct côté yiddish
def show_alignment(src_tok, tgt_tok, pairs, tgt_rtl=False):
    for s_i, t_i in pairs:
        t_word = tgt_tok[t_i]
        if tgt_rtl:
            t_word = get_display(t_word)
        print(f"{src_tok[s_i]:>15}  <->  {t_word}")

print("FR -> YI :")
show_alignment(fr_tok[i], yi_tok[i], result_yi["mwmf"], tgt_rtl=True)

print("FR -> RU :")
show_alignment(fr_tok[i], ru_tok[i], result["mwmf"])

FR -> YI :


NameError: name 'fr_tok' is not defined

### 4b. eflomal — un aligneur statistique classique (dans l'esprit de fast_align/IBM)

Contrairement à SimAlign, un aligneur statistique a besoin d'un **corpus d'une taille raisonnable** pour estimer ses probabilités : sur 5 phrases-jouets, ne vous attendez pas à un bon résultat — c'est justement intéressant à observer et à discuter. Sur votre vrai corpus (des centaines/milliers de lignes), le résultat sera bien plus représentatif.

In [ ]:
!pip -q install eflomal
import eflomal

src_lines = [" ".join(t) for t in fr_tok]
tgt_lines = [" ".join(t) for t in ru_tok]

al = eflomal.Aligner()
with open("src.txt", "w", encoding="utf-8") as f: f.write("\n".join(src_lines))
with open("tgt.txt", "w", encoding="utf-8") as f: f.write("\n".join(tgt_lines))

with open("src.txt", encoding="utf-8") as fsrc, open("tgt.txt", encoding="utf-8") as ftgt, \
     open("align.out", "w", encoding="utf-8") as fout:
    al.align(fsrc, ftgt, links_filename_fwd="align.out")

with open("align.out", encoding="utf-8") as f:
    for line in f:
        print(line.strip())

**Question :** comparez les résultats de SimAlign et d'eflomal sur la même paire de phrases. Lequel vous semble plus fiable ici, et pourquoi ? Est-ce que vous vous attendez à la même conclusion sur votre corpus complet ?

### 4c. Pour aller plus loin (optionnel)
[`awesome-align`](https://github.com/neulab/awesome-align) (Transformers, multilingue, fine-tunable) est une troisième option à essayer si vous avez le temps — installation un peu plus lourde (clonage du dépôt), mais souvent citée comme état de l'art. Comparez-la aux deux précédentes si vous allez plus loin.

*texte en italique*## 5. Alignement de phrases robuste aux problèmes de segmentation

Retour au problème de la section 1 : si la segmentation ne donne pas le même nombre de "phrases" dans chaque langue, un simple alignement ligne-à-ligne (ligne 1 avec ligne 1, ligne 2 avec ligne 2...) est faux. La solution utilisée en pratique : représenter chaque segment par un **vecteur (embedding) multilingue**, puis chercher le meilleur appariement par une recherche de type *programmation dynamique* (comme un alignement de séquences en bio-informatique), qui autorise les correspondances 1-à-plusieurs ou plusieurs-à-1.

On commence par le construire soi-même en quelques lignes, pour comprendre le principe — puis on le compare à une librairie prête à l'emploi.

In [ ]:
!pip -q install sentence-transformers
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer("sentence-transformers/LaBSE")

def embed(sents):
    return embedder.encode(sents, normalize_embeddings=True)

emb_fr = embed(fr_sents)
emb_ru = embed(ru_sents)

sim = emb_fr @ emb_ru.T  # matrice de similarité cosinus (déjà normalisés)
print(np.round(sim, 2))

In [ ]:
# Alignement 1-1 simple par programmation dynamique (maximiser la similarité cumulée,
# en avançant dans les deux textes en même temps -- une version simplifiée de l'idée
# utilisée par des outils comme Bertalign/Vecalign).
def align_dp(sim):
    n, m = sim.shape
    dp = np.full((n + 1, m + 1), -np.inf)
    dp[0, 0] = 0
    back = {}
    for a in range(n + 1):
        for b in range(m + 1):
            if a == 0 and b == 0:
                continue
            best, choice = -np.inf, None
            if a > 0 and b > 0 and dp[a-1, b-1] + sim[a-1, b-1] > best:
                best, choice = dp[a-1, b-1] + sim[a-1, b-1], (a-1, b-1, 'match')
            if a > 0 and dp[a-1, b] - 0.3 > best:  # pénalité de saut (insertion/omission)
                best, choice = dp[a-1, b] - 0.3, (a-1, b, 'skip_src')
            if b > 0 and dp[a, b-1] - 0.3 > best:
                best, choice = dp[a, b-1] - 0.3, (a, b-1, 'skip_tgt')
            dp[a, b] = best
            back[(a, b)] = choice
    # reconstruire le chemin
    path, a, b = [], n, m
    while (a, b) != (0, 0):
        prev = back[(a, b)]
        if prev[2] == 'match':
            path.append((prev[0], prev[1]))
        a, b = prev[0], prev[1]
    return list(reversed(path))

pairs = align_dp(sim)
for a, b in pairs:
    print(f"FR[{a}] <-> RU[{b}]")
    print("   ", fr_sents[a])
    print("   ", ru_sents[b])

### Comparaison avec une librairie dédiée : `bertalign`

[`bertalign`](https://pypi.org/project/bertalign/) implémente une version plus complète de la même idée (embeddings LaBSE + programmation dynamique en deux passes), et gère nativement les correspondances 1-à-plusieurs / plusieurs-à-1. C'est un outil pensé spécifiquement pour l'alignement de **textes littéraires**. Selon la version de Python de votre environnement Colab, l'installation peut échouer — c'est aussi une réalité du travail avec des outils NLP en évolution rapide ; en cas d'échec, comparez plutôt vos résultats de `align_dp` ci-dessus entre les trois paires de langues.

In [ ]:
try:
    !pip -q install bertalign
    from bertalign import Bertalign
    aligner_bt = Bertalign("\n".join(fr_sents), "\n".join(ru_sents))
    aligner_bt.align_sents()
    for src, tgt in aligner_bt.result:
        print(src, "<->", tgt)
except Exception as e:
    print("bertalign indisponible dans cet environnement :", e)
    print("-> comparez plutôt vos résultats de align_dp ci-dessus.")

## Références
- SimAlign — https://github.com/cisnlp/simalign
- eflomal — https://github.com/robertostling/eflomal
- awesome-align — https://github.com/neulab/awesome-align
- wtpsplit (segmentation multilingue) — https://github.com/segment-any-text/wtpsplit
- bertalign (alignement de phrases pour textes littéraires) — https://pypi.org/project/bertalign/
- python-bidi — https://github.com/MeirKriheli/python-bidi